# TernaryDense — storage & inference demo

Companion notebook for **keras-team/keras #22960** — `keras.layers.TernaryDense`, a drop-in `Dense` whose weights live in `{-1, 0, +1}` (BitNet b1.58, [arXiv:2402.17764](https://arxiv.org/abs/2402.17764)).

This notebook shows the concrete savings the PR introduces:

1. **Train a `TernaryDense`** and confirm every weight is exactly `-1`, `0`, or `+1`.
2. **Freeze for export** with `layer.quantize("ternary")` — outputs are *unchanged* (the frozen values equal the training forward value), and the kernel packs to **~1.58 bits/weight**.
3. **Measure storage density** vs `float32` / `int8` / `int4` on a realistic kernel.
4. **Post-training quantization** of a regular `Dense` via `layer.quantize("ternary")`.
5. **Sparseskip math** — why ternary turns the matmul into additions only (and the honest caveat about BLAS).

> Runs on Colab / Kaggle out of the box. Install the PR branch first (next cell).

In [ ]:
# Install the PR branch. On Colab/Kaggle this gives you keras with TernaryDense.
# (Swap to the merged keras release once #22960 lands.)
!pip install -q "git+https://github.com/simeon-kepp/keras@feat/layers/ternary-dense"

import numpy as np
import keras
from keras.layers import TernaryDense, Dense
from keras.quantizers import pack_ternary, unpack_ternary

print("keras", keras.__version__)
print("TernaryDense available:", hasattr(keras.layers, "TernaryDense"))

## 1. Train a `TernaryDense`

The layer keeps a float `kernel` that the optimizer updates, but on every forward pass it is *ternarized* on the fly (`sign(w)` where `|w| > threshold`, else `0`). With the default `threshold=None` the boundary is `0.5 * mean(|kernel|)` — the BitNet b1.58 §3.1 rule. Training uses a Straight-Through Estimator so gradients still reach the float weights.

In [ ]:
inputs = keras.Input((64,))
x = TernaryDense(128)(inputs)      # <- the new layer
outputs = TernaryDense(10, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Toy data
X = np.random.RandomState(0).randn(2000, 64).astype("float32")
y = np.random.RandomState(1).randint(0, 10, size=2000)
model.fit(X, y, epochs=3, batch_size=32, verbose=0)

# Inspect the effective (ternarized) kernel of the first layer
layer = model.layers[1]
kern = layer.kernel.numpy()
abs_k = np.abs(kern)
thr = 0.5 * abs_k.mean()
tern = np.where(abs_k > thr, np.sign(kern), 0.0)
print("effective weight values:", np.unique(tern))
print("zero fraction: %.3f" % (tern == 0).mean())

## 2. Freeze for export — `quantize("ternary")`

After training, `quantize("ternary")` freezes the ternarized kernel into a packed `uint8` representation. **Crucial property:** the packed values are exactly the layer's training forward value, so calling the layer before and after `quantize` gives identical outputs — no accuracy regression from freezing.

In [ ]:
x_test = np.random.RandomState(2).randn(8, 64).astype("float32")

out_before = model.predict(x_test, verbose=0)
model.layers[1].quantize("ternary")   # freeze first layer
model.layers[2].quantize("ternary")   # freeze second layer
out_after = model.predict(x_test, verbose=0)

max_diff = np.max(np.abs(out_before - out_after))
print("max |out_before - out_after| = %.2e  (should be ~0)" % max_diff)

# Save / load round-trip works on the packed form
model.save("/tmp/ternary_model.keras")
reloaded = keras.models.load_model("/tmp/ternary_model.keras")
out_reload = reloaded.predict(x_test, verbose=0)
print("max |out_after - out_reloaded| = %.2e  (should be ~0)" % np.max(np.abs(out_after - out_reload)))

## 3. Storage density

Five trits pack into one byte (`3**5 == 243 <= 256`), so the kernel lands at the information-theoretic floor of `log2(3) ≈ 1.58` bits/weight — strictly denser than `int8` (8) or `int4` (4).

The next cell reproduces the exact packing from `keras.quantizers.pack_ternary` on a realistic kernel and reports the real byte counts.

In [ ]:
rng = np.random.RandomState(0)
input_dim, units = 1024, 4096

# Simulate a trained ternary kernel (BitNet b1.58 thresholding)
raw = rng.randn(input_dim, units)
thr = 0.5 * np.mean(np.abs(raw))
tern = np.where(np.abs(raw) > thr, np.sign(raw), 0).astype(np.int8)

packed, packed_shape, orig_len = pack_ternary(tern, axis=0)
unpacked = unpack_ternary(packed, orig_len, axis=0)
print("round-trip exact:", np.array_equal(tern, unpacked))

float32_bytes = input_dim * units * 4
int8_bytes    = input_dim * units
int4_bytes    = input_dim * units // 2
ternary_bytes = int(np.prod(packed_shape))     # uint8 = 1 byte per packed cell
bpw = (ternary_bytes * 8) / (input_dim * units)

print("\n=== storage for a %dx%d kernel ===" % (input_dim, units))
print("float32 : %10,d bytes  32.00 bits/weight" % float32_bytes)
print("int8    : %10,d bytes   8.00 bits/weight" % int8_bytes)
print("int4    : %10,d bytes   4.00 bits/weight" % int4_bytes)
print("ternary : %10,d bytes   %.4f bits/weight (log2(3)=%.4f)" % (ternary_bytes, bpw, np.log2(3)))
print("\n=> %.2fx smaller than float32, %.2fx than int4, %.2fx than int8" % (32/bpw, 4/bpw, 8/bpw))

## 4. Post-training quantization of a regular `Dense`

You don't have to train with `TernaryDense`. A standard `Dense` can be frozen to the same packed ternary format after training via `layer.quantize("ternary")` (no STE — pure post-training pack).

In [ ]:
d = Dense(256, activation="relu")
d.build((None, 64))
d.kernel.assign(rng.randn(64, 256).astype("float32"))

xt = rng.randn(4, 64).astype("float32")
out_dense = d(xt).numpy()
d.quantize("ternary")            # freeze to packed ternary
out_tern = d(xt).numpy()

# For a *trained* Dense this is an approximation, so outputs shift slightly;
# the point is the storage win + that inference still runs from the packed kernel.
print("Dense quantized to ternary mode. dtype_policy:", d.dtype_policy.name)
print("max |dense - ternary| = %.4f" % np.max(np.abs(out_dense - out_tern)))
print("packed kernel shape:", d._packed_kernel.shape, "dtype:", d._packed_kernel.dtype)

## 5. Sparseskip — multiply-free inference (the honest part)

A ternary kernel makes the matmul structurally **multiply-free**: `output = scale * (inputs @ pos_mask - inputs @ neg_mask)`, where `pos_mask`/`neg_mask` are the `+1`/`-1` weights. Zero weights are simply skipped. So a kernel that is, say, 69% zeros does ~31% of the additions and **none** of the multiplications.

**Caveat (stated plainly in the PR):** a stock framework matmul (BLAS) still computes the zeros, so *this* PR delivers the storage win and an inference path that reaches parity by unpack-then-matmul. The actual compute speedup lives in a native ternary kernel that reads the packed format directly — a deliberate follow-up, not claimed here.

In [ ]:
nnz = np.mean(tern != 0)
print("nonzero fraction: %.3f" % nnz)
print("sparseskip matmul does %.3f of the additions, ZERO multiplications" % nnz)
print("-> a %.0f%%-zero kernel skips %.0f%% of the adds" % ((1-nnz)*100, (1-nnz)*100))

# Demonstrate the multiply-free form equals a normal matmul on the unpacked ternary kernel
x = rng.randn(16, input_dim).astype("float32")
k = unpack_ternary(packed, orig_len, axis=0).astype("float32")
normal = x @ k
pos = (k == 1).astype("float32")
neg = (k == -1).astype("float32")
sparseskip = x @ pos - x @ neg
# Two matmuls vs one accumulate slightly different rounding; assert up to fp tolerance.
print("\nmultiply-free form == normal matmul (within float rounding):",
      np.allclose(normal, sparseskip, atol=1e-2))